## Quiet the logs

`scarlets/utils/RedisLogger.py` calls `logging.basicConfig(level=DEBUG)` at
**import time**, which sets the *root* logger to DEBUG for the whole kernel -
so every library that logs (redis-py, httpx, asyncio) starts emitting DEBUG
into your cell output as soon as anything imports `scarlets`.

Run this first. It configures the root logger before `scarlets` gets the
chance, which makes that `basicConfig` call a no-op (`basicConfig` does
nothing when the root logger already has handlers).

If you ever see lines like `Failed to enable maintenance notifications:
unknown subcommand 'MAINT_NOTIFICATIONS'` - those are harmless. redis-py 8.x
probes for a Redis 8 maintenance feature on every new connection, Redis 7.x
replies "unknown subcommand", and the client logs it at DEBUG and carries on
with a perfectly normal connection.

In [1]:
import logging

# Claim the root logger first. logging.basicConfig() does nothing if the
# root logger already has handlers, so configuring it here makes
# RedisLogger's own basicConfig(level=DEBUG) - which runs when scarlets is
# imported further down - a no-op instead of a takeover.
logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")

# Belt and braces: if something already imported scarlets in this kernel
# (a re-run, or an earlier notebook), the call above was itself the no-op,
# so set the level explicitly too.
logging.getLogger().setLevel(logging.WARNING)
for _noisy in ("redis", "httpx", "httpx2", "httpcore", "httpcore2",
               "urllib3", "asyncio", "mcp"):
    logging.getLogger(_noisy).setLevel(logging.WARNING)

print("log level: WARNING (set INFO or DEBUG here if you want the detail back)")

log level: WARNING (set INFO or DEBUG here if you want the detail back)


# 14 — One call, a whole computation

Every earlier notebook spells the steps out in the prompt: agree a
representation, then sum, then combine. That works, but it puts the
**ordering in the prompt** — and a prompt that forgets a step produces a
confident wrong answer rather than an error.

Here the head asks for a variance and nothing else. The ordering lives in
the skill:

```
variance = [ agree_representation,
             mean,                      <- itself a compound
             sum_core(transform=square),
             combine "s2/n - mean**2" ]
```

Two properties are worth watching, because both are invisible when they
break:

**One consensus round for the whole nested plan.** `variance` agrees the
columns, and when it calls `mean`, the ambient namespace carries `columns`
down so `mean`'s own agreement step is skipped. If consensus runs twice,
composition is not actually sharing state — it is just hiding calls.

**`n` comes from `variance`'s own sum, not from `mean`.** A sub-compound
hands back only its declared result; its internal namespace does not leak
upward. `mean` returns a mean and nothing else.

The arithmetic is per-column throughout. `combine` carries vectors, so one
call handles every column at once rather than one call per column.


## Parameters

In [2]:
import os, time, json, subprocess, random

HEAD_APP_ID   = os.environ.get("HEAD_APP_ID", "scarlet-agent-head")
WORKER_APP_ID = os.environ.get("WORKER_APP_ID", "scarlet-agent-worker")

HEAD_NODE    = os.environ.get("HEAD_NODE_ADDRESS", "head")
WORKER_NODES = [n.strip() for n in
                os.environ.get("WORKER_NODE_ADDRESSES", "worker1,worker2,worker3,worker4").split(",")
                if n.strip()]

HEAD_BUS      = os.environ.get("HEAD_BUS", f"{HEAD_APP_ID}_headagent")
HEAD_HOST     = os.environ.get("HEAD_HOST", "agent-head")
HEAD_MCP_PORT = os.environ.get("HEAD_MCP_PORT", "8090")

_explicit_ids = os.environ.get("EXPECTED_AGENT_IDS", "").strip()
if _explicit_ids:
    EXPECTED_AGENT_IDS = {i.strip() for i in _explicit_ids.split(",") if i.strip()}
    HEAD_AGENT_ID      = f"{HEAD_APP_ID}_{HEAD_NODE}"
    WORKER_AGENT_IDS   = EXPECTED_AGENT_IDS - {HEAD_AGENT_ID}
else:
    HEAD_AGENT_ID      = f"{HEAD_APP_ID}_{HEAD_NODE}"
    WORKER_AGENT_IDS   = {f"{WORKER_APP_ID}_{n}" for n in WORKER_NODES}
    EXPECTED_AGENT_IDS = {HEAD_AGENT_ID} | WORKER_AGENT_IDS

GUSTAVO_MANAGED_AGENTS = os.environ.get("GUSTAVO_MANAGED_AGENTS", "false").lower() == "true"
COMPOSE_FILE = os.environ.get("COMPOSE_FILE_PATH", "/workspace/docker-compose.yml")

print(f"head app id   : {HEAD_APP_ID}")
print(f"worker app id : {WORKER_APP_ID}")
print(f"workers       : {WORKER_NODES}")
print(f"expecting     : {sorted(EXPECTED_AGENT_IDS)}")
print(f"fleet managed : {'Gustavo' if GUSTAVO_MANAGED_AGENTS else 'this notebook (docker compose)'}")

head app id   : loop-agent-head
worker app id : loop-agent-worker
workers       : ['worker1', 'worker2', 'worker3', 'worker4']
expecting     : ['loop-agent-head_head', 'loop-agent-worker_worker1', 'loop-agent-worker_worker2', 'loop-agent-worker_worker3', 'loop-agent-worker_worker4']
fleet managed : this notebook (docker compose)


## Generate each worker's matrix data and its local config

Four workers, four different shapes, four different column sets. The
intersection is never written down anywhere — it has to be discovered.

`HOME` for each worker points into this shared volume, so
`~/.scarlet/config.yaml` resolves to a path this notebook can write while
the worker still reads it from its real default location.


In [3]:
import numpy as np
import pandas as pd
import yaml

DATA_DIR = os.environ.get("DATA_DIR", "/data")

# torque_nm is deliberately held by only two of the four workers. Nothing
# in the paths or the source names says which - the head has to ask.
LAYOUT = {
    "worker1": ("plant_alpha/turbine_vibration.csv",  "turbine_alpha_vibration",  80,
                ["vibration_rms", "bearing_temp_c", "shaft_speed_rpm", "torque_nm", "power_kw"]),
    "worker2": ("feeds/grid_telemetry.csv",           "grid_telemetry_feed",     100,
                ["vibration_rms", "bearing_temp_c", "shaft_speed_rpm", "power_kw"]),
    "worker3": ("archive/2026/q3/unit7_sensors.csv",  "unit7_archive_q3",        120,
                ["vibration_rms", "bearing_temp_c", "shaft_speed_rpm", "torque_nm", "power_kw",
                 "ambient_temp_c"]),
    "worker4": ("raw_drop/sensor_export_FINAL.csv",   "bulk_sensor_drop",         90,
                ["vibration_rms", "shaft_speed_rpm", "power_kw"]),
}

PARAMS = {
    "vibration_rms":   (2.5,    0.6),
    "bearing_temp_c":  (65.0,   8.0),
    "shaft_speed_rpm": (1800.0, 120.0),
    "torque_nm":       (340.0,  45.0),
    "power_kw":        (64.0,   11.0),
    "ambient_temp_c":  (22.0,   4.0),
}

for i, (node, (relpath, source_name, rows, cols)) in enumerate(LAYOUT.items(), start=1):
    rng = np.random.default_rng(seed=i * 1000)
    start = pd.Timestamp("2026-01-01T00:00:00")
    frame = {"timestamp": [start + pd.Timedelta(minutes=k) for k in range(rows)]}
    for c in cols:
        mean, sd = PARAMS[c]
        frame[c] = np.clip(rng.normal(mean, sd, size=rows), 0.0, None)
    df = pd.DataFrame(frame)

    gap_cols = list(rng.permutation(cols))[:3]
    gap_rows = rng.choice(rows, size=3, replace=False)
    for r, c in zip(gap_rows, gap_cols):
        df.at[r, c] = np.nan

    path = os.path.join(DATA_DIR, relpath)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df.to_csv(path, index=False)

    scarlet_dir = os.path.join(DATA_DIR, node, ".scarlet")
    os.makedirs(scarlet_dir, exist_ok=True)
    actual_gaps = sorted(df.columns[df.isna().any()].tolist())
    with open(os.path.join(scarlet_dir, "config.yaml"), "w") as fh:
        yaml.safe_dump({"sources": [{
            "name": source_name, "type": "csv", "mode": "local",
            "description": (f"{rows} rows. Columns: {', '.join(cols)}. "
                            f"gaps in columns: {', '.join(actual_gaps)}"),
            "path": path,
        }]}, fh, sort_keys=False)

    has = "has torque_nm" if "torque_nm" in cols else "NO torque_nm"
    print(f"{source_name:26s} {rows:>4} rows  {has}")

# Ground truth. The fleet will agree on the columns every worker holds -
# the intersection - so compute over exactly those.
_sets = [set(v[3]) for v in LAYOUT.values()]
_common = sorted(set.intersection(*_sets))
_parts = [pd.read_csv(os.path.join(DATA_DIR, v[0]))[_common].dropna().values
          for v in LAYOUT.values()]
_all = np.vstack(_parts)
_n = _all.shape[0]
_mean = _all.sum(axis=0) / _n
_var = (_all ** 2).sum(axis=0) / _n - _mean ** 2
print(f"\ncommon columns the fleet must discover: {_common}")
print(f"expected n = {_n}")
for _c, _m, _v in zip(_common, _mean, _var):
    print(f"  {_c:18s} mean={_m:12.4f}  variance={_v:12.4f}")


turbine_alpha_vibration      80 rows  has torque_nm
grid_telemetry_feed         100 rows  NO torque_nm
unit7_archive_q3            120 rows  has torque_nm
bulk_sensor_drop             90 rows  NO torque_nm

common columns the fleet must discover: ['power_kw', 'shaft_speed_rpm', 'vibration_rms']
expected n = 380
  power_kw           mean=     63.7831  variance=    113.9624
  shaft_speed_rpm    mean=   1801.6797  variance=  14471.2529
  vibration_rms      mean=      2.5474  variance=      0.3575


## Bring the fleet up

In [4]:
#now we are going to bring the workers up. these can be made fully decentralized using Gustavo!

AGENT_SERVICES = ["agent-head"] + [f"agent-{n}" for n in WORKER_NODES]

def compose(*args):
    result = subprocess.run(
        ["docker", "compose", "-f", COMPOSE_FILE, "--profile", "agents", *args],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(
            f"`docker compose {' '.join(args)}` failed with exit "
            f"{result.returncode} - see the Compose output above."
        )
    return result

if GUSTAVO_MANAGED_AGENTS:
    print("Gustavo owns this fleet - not launching anything.")
    print("If the wait below times out, check the Gustavo deployment.")
else:
    missing_llm = [k for k in ("LLM_BASE_URL", "LLM_MODEL")
                   if not os.environ.get(k, "").strip()]
    if missing_llm:
        raise RuntimeError(
            f"{', '.join(missing_llm)} not set - the agent head cannot start "
            f"without them.\nSet these in examples/notebooks/.env and restart "
            f"the stack (docker compose up -d), then re-run this notebook."
        )
    
    print("bringing up:", ", ".join(AGENT_SERVICES))
    #compose("up", "-d", "--no-deps", *AGENT_SERVICES)
    print("containers started - waiting for them to register below")

bringing up: agent-head, agent-worker1, agent-worker2, agent-worker3, agent-worker4
containers started - waiting for them to register below


## Wait for the fleet to come online

In [5]:
from scarlets.messaging import Messenger

FRESH_SECONDS = 90

_probe = Messenger(HEAD_BUS, agentId="notebook_probe")

def live_agents():
    now = time.time()
    return {
        agent_id
        for agent_id, record in _probe.GatherStatus().items()
        if now - record.get("ts", 0) < FRESH_SECONDS
    }

def wait_until_live(timeout):
    deadline = time.time() + timeout
    while True:
        online  = live_agents()
        missing = EXPECTED_AGENT_IDS - online
        if not missing:
            print(f"fleet online: {sorted(EXPECTED_AGENT_IDS)}")
            return online
        if time.time() >= deadline:
            where = ("the Gustavo deployment" if GUSTAVO_MANAGED_AGENTS
                     else f"docker compose -f {COMPOSE_FILE} --profile agents logs")
            raise TimeoutError(
                f"after {timeout}s these agents are still not live on {HEAD_BUS!r}: "
                f"{sorted(missing)}\nSeen on the bus: {sorted(live_agents())}\n"
                f"Check {where}."
            )
        time.sleep(2)

wait_until_live(timeout=180 if GUSTAVO_MANAGED_AGENTS else 120)

fleet online: ['loop-agent-head_head', 'loop-agent-worker_worker1', 'loop-agent-worker_worker2', 'loop-agent-worker_worker3', 'loop-agent-worker_worker4']


{'__composer_api__',
 'loop-agent-head_head',
 'loop-agent-worker_worker1',
 'loop-agent-worker_worker2',
 'loop-agent-worker_worker3',
 'loop-agent-worker_worker4',
 'notebook_probe'}

## Ask the head for per-column statistics

The head is given an objective and nothing else — no source, no path, no
column. It has to run the consensus step first, because `sum` will refuse
to aggregate matrices of differing width.

Mean and variance per column come from the same two-sums composition 08
used, except every quantity is now a vector: `sum(x)` and `sum(x^2)` are
per-column, so mean and variance come out per-column too.


In [6]:
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client


async def ask_scarlet_agent(message: str) -> str:
    url = f"http://{HEAD_HOST}:{HEAD_MCP_PORT}/mcp"
    async with streamable_http_client(url) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            result = await session.call_tool("ask_scarlet_agent", {"message": message})
            return result.content[0].text


# No step list. The prompt names a quantity, not a procedure - which is the
# whole point of a compound skill. Compare notebook 10, where the same
# computation needed four numbered steps spelled out here.
answer = await ask_scarlet_agent(
    "Compute the per-column variance of the sensor measurements held across "
    "the fleet, and the per-column mean. Use the variance and mean skills. "
    "Report, for each column: its name, its mean, and its variance. State n "
    "once. Do not name any data source or column yourself - the workers know "
    "their own data."
)
print(answer)


The per-column statistics across the fleet (n = 380) are:

- **power_kw**: mean = 63.78, variance = 113.96  
- **shaft_speed_rpm**: mean = 1801.68, variance = 14471.25  
- **vibration_rms**: mean = 2.55, variance = 0.36


In [7]:
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client


async def ask_scarlet_agent(message: str) -> str:
    url = f"http://{HEAD_HOST}:{HEAD_MCP_PORT}/mcp"
    async with streamable_http_client(url) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            result = await session.call_tool("ask_scarlet_agent", {"message": message})
            return result.content[0].text


# The ordering instruction is load-bearing. `sum` fails deliberately on
# mismatched widths rather than truncating to the narrowest, so the
# agreement has to happen first and its exact column list has to be passed
# through to both sum calls - otherwise each worker reads whatever it
# happens to hold and the shapes will not line up.
answer = await ask_scarlet_agent(
    "Can you tell me if any of the workers have any sources pertaining to tasks? Try to do a query_feature for *_tasks? you can try different combinations."
    "Also tell me if there are any sources relating to turbine, telemetry or vibration information? repeat the same process as before."
)
print(answer)


Error executing tool ask_scarlet_agent


## Teardown

In [8]:
if GUSTAVO_MANAGED_AGENTS:
    print("Gustavo owns this fleet - leaving it alone.")
else:
    #compose("rm", "-s", "-f", *AGENT_SERVICES)
    print("agent containers stopped and removed")

agent containers stopped and removed
